# CSE 4095 — Class 2: Classification
Use AI as a coding agent, but inspect and explain every major decision.

## 1. Load the Class 1 dataset
Upload `class1_student_habits.csv`, import Pandas, load it as `df`, and inspect the first rows.

In [1]:
import pandas as pd

# Only blank cells count as missing, so "None" stays a real
# prior_programming_experience answer instead of turning into NaN.
df = pd.read_csv("class1_student_habits.csv", keep_default_na=False, na_values=[""])

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Rows: 250, Columns: 10


,student_id,study_hours_per_week,sleep_hours_per_night,attendance_rate,practice_problems_per_week,prior_programming_experience,study_group,commute_minutes,quiz_average,exam_score
0,S001,1.3,7.7,71.8,8,Strong,No,0,67.0,64.6
1,S002,9.4,6.6,74.2,14,None,Yes,52,74.8,81.5
2,S003,11.2,6.1,70.3,15,Some,No,7,99.1,92.0
3,S004,10.8,7.6,77.1,16,Strong,No,12,86.4,100.0
4,S005,8.0,5.6,89.7,15,None,Yes,12,75.7,68.5


In [2]:
# Missing values by column (blank cells only)
df.isna().sum()

student_id                       0
study_hours_per_week             9
sleep_hours_per_night           12
attendance_rate                  0
practice_problems_per_week       0
prior_programming_experience     7
study_group                      0
commute_minutes                  0
quiz_average                     0
exam_score                       0
dtype: int64

## 2. Create the target
Create `high_exam_score`: Yes if `exam_score >= 80`, otherwise No. Then inspect class counts.

In [3]:
TARGET = "high_exam_score"
df[TARGET] = (df["exam_score"] >= 80).map({True: "Yes", False: "No"})

pd.DataFrame({
    "count": df[TARGET].value_counts(),
    "share": df[TARGET].value_counts(normalize=True).round(3),
})

,count,share
high_exam_score,,
No,151,0.604
Yes,99,0.396


## 3. Choose X and y
Use only legitimate features. Explain in Markdown why `student_id` and `exam_score` are excluded.

In [4]:
RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
]
CATEGORICAL_FEATURES = [
    "prior_programming_experience",
    "study_group",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES]
y = df[TARGET]
print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (250, 6), y shape: (250,)


### Which columns would make the task artificially easy?

| Column | Verdict | Why |
|---|---|---|
| `exam_score` | **Leakage** | The target is computed directly from it. A model would just learn "score >= 80". |
| `quiz_average` | **Leakage** (for this task) | Measures the same skill as the exam (r = 0.77, far above any habit) and is not known when we only have habit data. |
| `student_id` | Excluded | An identifier with no real signal. A model could memorize individual students instead of learning patterns. |
| `commute_minutes` | Excluded | Not leakage, but almost no signal (r = 0.07) and not one of the intended features. |

The six features kept are all study habits or background known before the exam.

**Process leakage:** fitting imputers or scalers on the full dataset before splitting lets test rows influence training. The pipeline in Sections 5 and 6 prevents this.

## 4. Baseline
Calculate the majority-class baseline accuracy before training a model.

In [5]:
class_shares = y.value_counts(normalize=True)
majority_class = class_shares.idxmax()

print(f"Majority class: {majority_class}")
print(f"Always guessing '{majority_class}' is right {class_shares.max():.1%} of the time.")

Majority class: No
Always guessing 'No' is right 60.4% of the time.


**Why calculate the baseline first?**
- Accuracy means nothing without a reference point. 65% sounds fine until you learn that always guessing "No" gets 60%.
- The classes are imbalanced (99 Yes / 151 No). The baseline finds **0%** of the high scorers and still gets 60% accuracy, so accuracy alone can hide a useless model.
- A model that scores below the baseline points to a bug, such as flipped labels or a broken feature.

## 5. Train/test split and preprocessing
Ask your AI agent for a pipeline that handles numerical/categorical missing values and one-hot encoding.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "Yes": [(y_train == "Yes").sum(), (y_test == "Yes").sum()],
    "No": [(y_train == "No").sum(), (y_test == "No").sum()],
    "Yes_share": [(y_train == "Yes").mean().round(3), (y_test == "Yes").mean().round(3)],
}, index=["train", "test"])

,rows,Yes,No,Yes_share
train,200,79,121,0.395
test,50,20,30,0.400


**Why stratify?** The classes are imbalanced (40% Yes) and the test set is only 50 rows. Across 1,000 unstratified splits, the test set had anywhere from 11 to 30 "Yes" students, and 18% of splits were off by more than 4. `stratify=y` keeps both sets at about 40% Yes, so scores reflect the model and not a lucky split. `random_state` makes the split repeatable.

In [7]:
# Baseline scored the same way a real model will be:
# learn the majority class from TRAIN, score on TEST.
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_accuracy = baseline_model.score(X_test, y_test)

print(f"Baseline always predicts: {baseline_model.predict(X_test)[0]}")
print(f"Baseline test accuracy:   {baseline_accuracy:.1%}")

Baseline always predicts: No
Baseline test accuracy:   60.0%


In [8]:
# What the preprocessing has to handle, looked at on TRAIN only
print("Missing values after the split:")
print(pd.DataFrame({"train": X_train.isna().sum(), "test": X_test.isna().sum()}))

print("\nFill values the imputers will learn (from X_train only):")
print(X_train[NUMERIC_FEATURES].median().round(2).to_string())
print(X_train[CATEGORICAL_FEATURES].mode().iloc[0].to_string())

print("\nCategories seen in training:")
for column in CATEGORICAL_FEATURES:
    train_categories = sorted(X_train[column].dropna().unique())
    test_only = set(X_test[column].dropna()) - set(train_categories)
    print(f"  {column}: {train_categories}, only in test: {sorted(test_only) or 'none'}")

Missing values after the split:
                              train  test
study_hours_per_week              6     3
sleep_hours_per_night            11     1
attendance_rate                   0     0
practice_problems_per_week        0     0
prior_programming_experience      5     2
study_group                       0     0

Fill values the imputers will learn (from X_train only):
study_hours_per_week           7.40
sleep_hours_per_night          7.10
attendance_rate               85.75
practice_problems_per_week    16.00
prior_programming_experience    Some
study_group                      Yes

Categories seen in training:
  prior_programming_experience: ['None', 'Some', 'Strong'], only in test: none
  study_group: ['No', 'Yes'], only in test: none


### Pipeline concepts

- **Transformer vs estimator:** transformers (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`) have `fit` (learn from data) and `transform` (apply it). Estimators (`LogisticRegression`) have `fit` and `predict`.
- **`Pipeline`:** chains steps in order. `fit` runs `fit_transform` on each transformer using the training data, then fits the model. `predict` only runs `transform` with what was already learned, then predicts. Test data never refits anything, which prevents process leakage.
- **`ColumnTransformer`:** sends numeric columns down one branch and categorical columns down another, then puts the results side by side (4 numeric + 3 experience + 2 study-group = 9 columns). Columns you don't list are dropped.
- **`SimpleImputer`:** fills missing values with a statistic learned during `fit`. The mean is pulled by outliers and the median is not. Text has no mean or median.
- **`StandardScaler`:** rescales each numeric column to mean 0 and standard deviation 1. Logistic regression penalizes every coefficient equally, so without scaling, columns with large units would be penalized unevenly. Scaled coefficients can also be compared with each other.
- **`OneHotEncoder`:** turns each category into its own 0/1 column. Mapping None/Some/Strong to 0/1/2 would assume the gaps between them are equal. `handle_unknown` controls what happens when the test set has a category train never saw.
- **Step names** (`"impute"`, `"classifier"`) are used for access (`model.named_steps[...]`), become column prefixes (`numeric__study_hours_per_week`), and are used for tuning (`classifier__C`).

**Your turn:** fill in Steps 1 to 3 below. The cell still runs while they're blank.

In [9]:
# Step 1: numeric branch (runs top to bottom on NUMERIC_FEATURES)
#   ("impute", SimpleImputer(strategy=???))
#       Hint: mean or median? Remember the practice_problems outlier (78).
#   ("scale", StandardScaler())
numeric_pipeline = Pipeline(steps=[
    # TODO: add the two (name, transformer) tuples
])

# Step 2: categorical branch (runs top to bottom on CATEGORICAL_FEATURES)
#   ("impute", SimpleImputer(strategy=???))
#       Hint: text has no mean or median. Which strategy works on text?
#   ("encode", OneHotEncoder(handle_unknown=???))
#       Hint: what should happen if the test set has a category train never saw?
categorical_pipeline = Pipeline(steps=[
    # TODO: add the two (name, transformer) tuples
])

# Step 3: send each column list to its branch
#   Each entry is (name, pipeline, list_of_columns).
preprocessor = ColumnTransformer(transformers=[
    # TODO: ("numeric", ..., ...)
    # TODO: ("categorical", ..., ...)
])

## 6. Logistic regression
Fit the model using training data only.

In [10]:
# Step 4: full model = preprocessing, then the classifier
#   ("preprocess", preprocessor)
#   ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
model = Pipeline(steps=[
    # TODO: add the two (name, step) tuples
])

# Step 5: fit on TRAIN only, then check your work.
# When Steps 1-4 are filled in, delete the "if" line and un-indent.
if model.steps:
    model.fit(X_train, y_train)

    transformed_train = model.named_steps["preprocess"].transform(X_train)
    print(f"Shape after preprocessing: {transformed_train.shape}  (expected: (200, 9))")
    print(f"Missing values left:       {pd.DataFrame(transformed_train).isna().sum().sum()}  (expected: 0)")
    print("Columns the model sees:")
    print(list(model.named_steps["preprocess"].get_feature_names_out()))

    test_accuracy = model.score(X_test, y_test)
    print(f"\nTest accuracy: {test_accuracy:.1%}  vs baseline {baseline_accuracy:.1%}")

## 7. Evaluate
Calculate accuracy, confusion matrix, precision, and recall. Compare accuracy with the baseline.

In [11]:
# TODO: evaluate test predictions


## 8. Inspect individual predictions
Create a small table with actual class, predicted class, and predicted probability.

In [12]:
# TODO: inspect predictions


## 9. Controlled leakage experiment
Temporarily include `exam_score` as a feature, compare results, and explain why this experiment is invalid as a real model.

In [13]:
# TODO: leakage experiment


## 10. Your experiment
Change one legitimate modeling decision. Write your prediction before running it, then compare results.

In [14]:
# TODO: student-designed experiment


## Conclusion
Write a short evidence-based conclusion and AI Reflection here.
